In [ ]:
!pip install -q langgraph groq gradio

In [ ]:
import gradio as gr
from typing import TypedDict
from langgraph.graph import StateGraph, END
from groq import Groq

In [ ]:
GROQ_API_KEY = "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"

client = Groq(api_key=GROQ_API_KEY)
MODEL_NAME = "llama-3.1-8b-instant"

In [ ]:
def call_llama(prompt):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

In [ ]:
class RocketState(TypedDict):
    temperature: float
    threshold: float
    action: str
    report: str

In [ ]:
def safety_reflex_node(state: RocketState):

    if state["temperature"] > state["threshold"]:
        state["action"] = "🚨 AUTOMATIC ENGINE SHUTDOWN INITIATED"
    else:
        state["action"] = "✅ Engine Operating Within Safe Limits"

    return state

In [ ]:
def llm_report_node(state: RocketState):

    prompt = f"""
    You are an Aerospace Safety Monitoring AI System.

    Generate a highly professional aerospace mission control report.

    Engine Temperature: {state['temperature']} °C
    Safety Threshold: {state['threshold']} °C
    System Action: {state['action']}

    Include:
    - Mission Status
    - Risk Level
    - Recommended Next Steps
    - Safety Certification Tone

    Use structured formatting and aerospace terminology.
    """

    state["report"] = call_llama(prompt)
    return state

In [ ]:
workflow = StateGraph(RocketState)

workflow.add_node("ReflexSafety", safety_reflex_node)
workflow.add_node("LLMReport", llm_report_node)

workflow.set_entry_point("ReflexSafety")

workflow.add_edge("ReflexSafety", "LLMReport")
workflow.add_edge("LLMReport", END)

app = workflow.compile()

In [ ]:
custom_css = """
body {
    background-color: #f3f6fb;
}
.gradio-container {
    font-family: 'Segoe UI', sans-serif;
}
.banner {
    white-space: nowrap;
    overflow: hidden;
    box-sizing: border-box;
}
.banner span {
    display: inline-block;
    padding-left: 100%;
    animation: marquee 12s linear infinite;
    font-weight: bold;
    color: #1e3a8a;
}
@keyframes marquee {
    0%   { transform: translate(0, 0); }
    100% { transform: translate(-100%, 0); }
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:

    gr.HTML("""
    <div class="banner">
        <span>🚀 Rocket Engine Safety Shutdown by Collins Aerospace 🚀</span>
    </div>
    """)

    gr.Markdown("## Aerospace Launch Safety Reflex Monitoring System")

    with gr.Row():
        with gr.Column(scale=1):
            temp_input = gr.Number(label="Engine Temperature (°C)", value=850)
            threshold_input = gr.Number(label="Safety Threshold (°C)", value=900)
            analyze_btn = gr.Button("Run Safety Check", variant="primary")

        with gr.Column(scale=1):
            output_box = gr.Markdown()

    def run_safety_check(temp, threshold):

        state = {
            "temperature": temp,
            "threshold": threshold,
            "action": "",
            "report": ""
        }

        result = app.invoke(state)
        return result["report"]

    analyze_btn.click(
        run_safety_check,
        inputs=[temp_input, threshold_input],
        outputs=output_box
    )

demo.launch(debug=True)